In [ ]:
# This notebook is meant to generate the threshold file used in simulation, taking the
# threshold file used for data as input. Because there are tile swaps in data, which is
# currently not taken care of in simulation, we have to manually swap the tile labels
# in simulation so that the channel-by-channel thresholds correspond to the same physical space. 

In [ ]:
import json
import numpy as np

In [ ]:
with open('../../data/proto_nd_flow/thresholds_2x2.json', 'r') as f:
    data = json.load(f)

In [ ]:
# Currently, the threshold file is written in terms of io_channel instead of tile_id, inconsistent 
# with other files. This should be corrected in the future so everything is consistent. 

# The below converts a tile number to a set of io_channels
def tile_id_to_channel_id(io_group, tile):
    l = []
    for i in np.arange(1,5) + 4*(tile-1):
        l.append((str(io_group),  "{:02d}".format(i)))
    return l
    


In [ ]:
# These numbers were obtained from Figure 13 and 14 of the charge geometry document
# https://docs.dunescience.org/cgi-bin/sso/RetrieveFile?docid=32440&filename=2x2_Demonstrator_Geometry_Description-2.pdf&version=2
# For (x,y) x is the io_group and y is the tile number. 
swaps = {}
swaps[(1,4)] = (1,8)
swaps[(1,7)] = (1,4)
swaps[(1,8)] = (1,7)

swaps[(5,7)] = (5,8)
swaps[(5,8)] = (5,7)

swaps[(7,5)] = (7,8)
swaps[(7,8)] = (7,5)
swaps[(8,1)] = (8,2)
swaps[(8,2)] = (8,1)

global_swaps = {}
for k, v in swaps.items():
    for old, new in zip(tile_id_to_channel_id(*k), tile_id_to_channel_id(*v)):
        global_swaps[old] = new

In [ ]:
# Probably not the best way to do this, but I do the replacement by taking the string position
# corresponding to the io_channel and manipulating that
swapped_dict = {}
all_count = 0
swapped_count = 0
for key, value in data.items():
    io_group = key[0]
    old_io_channel = key[2:4]
    # print(key, io_group, old_tile)

    if (io_group, old_io_channel) in global_swaps.keys():
        _, new_io_channel = global_swaps[(io_group, old_io_channel)]
        new_key = key[:2] + new_io_channel + key[4:]
        # print("Old key: {old_key}\nNew key: {new_key}\n".format(old_key = key, new_key=new_key))
        swapped_count +=1
    else:
        new_key = key
    all_count +=1
    swapped_dict[new_key] = value


In [ ]:
with open('../../data/proto_nd_flow/thresholds_2x2.swapped.json', 'w') as f:
    json.dump(swapped_dict, f)